## KDD: Recuento

Lee el esquema final de `data/final/` y verifica las columnas

### Imports

In [1]:
from pathlib import Path
import pyarrow.parquet as pq

---
### 1. Esquema final

In [ ]:
proyecto = Path.cwd()
while not (proyecto / "docker-compose.yml").exists() and proyecto != proyecto.parent:
    proyecto = proyecto.parent
procesos_dir = proyecto / "data" / "final"

assert procesos_dir.exists(), \
    f"No encuentro {procesos_dir}. Corre KDD/a_transformaciones.ipynb (Run All) antes que este notebook."

algun_parquet = next(procesos_dir.glob("anio=*/mes=*/*.parquet"))
schema = pq.read_schema(algun_parquet)

print(f"{len(schema.names)} columnas:")

for nombre, tipo in zip(schema.names, schema.types):
    print(f"  {nombre:20s} {tipo}")

21 columnas:
  ocid                 string
  fecha                timestamp[ns]
  periodo              string
  entidad              string
  region               string
  proveedor            string
  categoria            string
  metodo_detalle       string
  n_postores           int32
  postor_unico         int32
  monto_referencial    double
  monto_adjudicado     double
  monto_final          double
  tramo_monto          string
  descripcion          string
  texto                string
  tokens               list<element: string>
  shingles             list<element: string not null>
  unspsc               list<element: string>
  rubro                list<element: string>
  canasta              list<element: string>


### 3. Recuento de columnas por técnica

### 3.1 Similitud

- **Columnas:** `shingles`, `texto`, `entidad`, `fecha`. `shingles` (`generar_shingles`)

- `entidad`/`fecha` sirven para filtrar pares candidatos (misma entidad, fechas cercanas) antes de comparar similitud.

### 3.2 Vecinos aproximados

- **Columnas:** `tokens`, `monto_final`. `tokens` (`tokenizar`)

- Listo para vectorizar con TF-IDF; `monto_final` es el valor numérico que se compara entre los *k* vecinos más cercanos

### 3.3 Minería de flujos

- **Columnas:** `fecha` (orden del flujo), `entidad`, `proveedor`, `postor_unico`. `postor_unico` (`agregar_tramos_monto`)

- Permite contar con DGIM cuántos procesos de postor único hubo en una ventana reciente, y con Bloom/Count-Min si un par entidad-proveedor ya se vio antes

### 3.4 Reglas de asociación

- **Columnas:** `canasta` (`construir_canasta`)

- Cada proceso ya es una transacción de ítems `clave=valor`, lista para Apriori/FP-Growth sin
  transformación adicional: `cat=`, `met=`, `reg=`, `monto=`, `comp=postor_unico`/`comp=varios_postores`,
  `rubro=` (uno por código UNSPSC distinto) — detalle en `KDD/a_transformaciones.ipynb` §4.4

---
### 4. Uso de la data

Se lee directamente de `data/final/`

```python
procesos = spark.read.parquet("data/final")                              # los tres años
procesos_2025 = spark.read.parquet("data/final").where("anio = 2025")    # sólo 2025
```